# Step 7: Build the road network, health facilities and villages

**Run this in Google Colab** (*Runtime > Run all*). No GPU needed. It takes about 30 to 60 minutes, mostly downloading.

This step does **not** need the flood maps. It runs at the same time as Step 6.

**What it makes** (all in metres, EPSG:32646, the same projection as the flood maps):

| File | What it holds |
|---|---|
| `network.gpkg` | Road network: layers `nodes` (junctions), `edges` (road stretches, with `is_bridge`), `strips` (20 m wide strip around each stretch) |
| `facilities.gpkg` | Health facilities, one point each, with the junction they connect to |
| `villages.gpkg` | Settlements (groups of buildings), with people, district, junction and normal route length to the nearest facility |
| `districts.gpkg` | The three district boundaries |
| `summary_network.csv` | One row per district: villages, people, facilities, problems |

**Important choices** (all in `src/config.py`, never typed here):
- The network covers the whole study box, not just the three districts, so a village near a district border can use a hospital across the border.
- Walking-only paths (footways, steps) are left out. Tracks and paths are kept.
- Buildings within 100 m of each other form one settlement. A settlement more than 500 m from any road is marked "no road access".

At the end, copy the outputs to Google Drive and share the folder with Adityapratap.

In [ ]:
!pip install -q osmnx
!apt-get -qq install -y osmium-tool > /dev/null
!osmium --version | head -1

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/Matrixco123/cutoff-flood.git"
REPO = "/content/cutoff-flood"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
sys.path.insert(0, REPO)
from src import config
print("Project code loaded")

In [ ]:
import datetime, time, json, math
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
from shapely.geometry import box
from sklearn.cluster import DBSCAN

TODAY = datetime.date.today().isoformat()
RAW = "/content/raw"            # downloads, saved with dates
WORK = "/content/work"
OUT = "/content/outputs_network"
for d in (RAW, WORK, OUT):
    os.makedirs(d, exist_ok=True)

CRS = config.WORK_CRS
L, B, R, T = config.STUDY_BBOX


def osmium(*args):
    r = subprocess.run(["osmium", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr)


def road_class(h):
    return h[0] if isinstance(h, list) else h


def download(url, dest):
    if not os.path.exists(dest) or os.path.getsize(dest) == 0:
        subprocess.run(["wget", "-q", "-O", dest, url], check=True)
    print(f"{os.path.basename(dest)}: {os.path.getsize(dest) / 1e6:.0f} MB")

## 1. Download the data (dated copies)

- OpenStreetMap for northeast India from Geofabrik (about 100 MB)
- WorldPop 2020 population, India, 100 m (about 530 MB)
- Google Open Buildings v3 is read in section 4, streamed straight from Google's server

In [ ]:
PBF = f"{RAW}/osm_northeast_{TODAY}.osm.pbf"
download("https://download.geofabrik.de/asia/india/north-eastern-zone-latest.osm.pbf", PBF)

WORLDPOP_URL = ("https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/"
                "2020/BSGM/IND/ind_ppp_2020_constrained.tif")
WORLDPOP = f"{RAW}/worldpop_ind_2020_constrained.tif"
download(WORLDPOP_URL, WORLDPOP)

## 2. Road network for the whole study box

Steps: cut the study box out of the OSM file, keep only roads, build the network without merging road stretches, count each two-way road once, drop non-roads and walking-only paths, convert to metres.

In [ ]:
def build_network(roads_osm):
    """Undirected road network in metres, without excluded road types."""
    G = ox.graph_from_xml(roads_osm, simplify=False, retain_all=True)
    # Project first: project_graph returns a one-way (directed) graph, which would
    # make routes follow the drawing direction of each road.
    G = ox.project_graph(G, to_crs=CRS)
    G = ox.convert.to_undirected(G)
    drop = [(u, v, k) for u, v, k, d in G.edges(keys=True, data=True)
            if road_class(d.get("highway")) in config.EXCLUDED_ROAD_TYPES]
    G.remove_edges_from(drop)
    G.remove_nodes_from([n for n in list(G.nodes) if G.degree(n) == 0])
    assert not G.is_directed()
    return G


def is_bridge(b):
    """OSM bridge tag: 'yes', 'viaduct' etc. = bridge; missing or 'no' = not a bridge."""
    b = b[0] if isinstance(b, list) else b
    return isinstance(b, str) and b != "no"


def edges_table(G):
    """Road stretches as a table, with road class and bridge flag."""
    e = ox.graph_to_gdfs(G, nodes=False).reset_index()
    e["cls"] = e["highway"].apply(road_class)
    if "bridge" not in e.columns:
        e["bridge"] = None
    e["is_bridge"] = e["bridge"].apply(is_bridge)
    e["edge_id"] = np.arange(len(e))
    return e


box_pbf = f"{WORK}/study_box.osm.pbf"
osmium("extract", "-b", f"{L},{B},{R},{T}", PBF, "-o", box_pbf, "-O")
roads_osm = f"{WORK}/roads.osm"
osmium("tags-filter", box_pbf, "w/highway", "-o", roads_osm, "-O")

t0 = time.time()
G = build_network(roads_osm)
edges = edges_table(G)
nodes = ox.graph_to_gdfs(G, edges=False).reset_index()
print(f"Network: {G.number_of_nodes():,} junctions, {G.number_of_edges():,} road stretches, "
      f"{edges.length.sum() / 1000:,.0f} km, {edges.is_bridge.sum():,} bridge stretches "
      f"({time.time() - t0:.0f} s)")
print(edges.groupby("cls").length.sum().div(1000).sort_values(ascending=False).round(0).head(12))

## 3. Health facilities

Hospitals, clinics, doctors and anything tagged `healthcare`. Points closer than 50 m are merged (the same place is often mapped twice, as a point and as a building outline). Each facility is joined to its nearest road junction.

In [ ]:
HEALTH_TAGS = {"amenity": ["hospital", "clinic", "doctors"], "healthcare": True}


def load_facilities(health_osm, G):
    f = ox.features_from_xml(health_osm, tags=HEALTH_TAGS).reset_index()
    f = f.to_crs(CRS)
    f["geometry"] = f.geometry.centroid
    f["kind"] = f.get("amenity", pd.Series(index=f.index, dtype=object)).fillna(
        f.get("healthcare", pd.Series(index=f.index, dtype=object)))
    f["name"] = f.get("name", pd.Series(index=f.index, dtype=object))
    # merge points closer than FACILITY_MERGE_M, keep the one with a name (or the first)
    xy = np.c_[f.geometry.x, f.geometry.y]
    f["group"] = DBSCAN(eps=config.FACILITY_MERGE_M, min_samples=1).fit_predict(xy)
    f["has_name"] = f["name"].notna()
    f = f.sort_values("has_name", ascending=False).drop_duplicates("group")
    f = f[["name", "kind", "geometry"]].reset_index(drop=True)
    node, dist = ox.distance.nearest_nodes(G, f.geometry.x.values, f.geometry.y.values, return_dist=True)
    f["node"], f["snap_m"] = node, np.round(dist, 1)
    f["on_network"] = f["snap_m"] <= config.SNAP_MAX_M
    f["facility_id"] = np.arange(len(f))
    return gpd.GeoDataFrame(f, geometry="geometry", crs=CRS)


health_osm = f"{WORK}/health.osm"
osmium("tags-filter", box_pbf, "nwr/amenity=hospital,clinic,doctors", "nwr/healthcare", "-o", health_osm, "-O")
facilities = load_facilities(health_osm, G)
print(f"{len(facilities)} facilities, {(~facilities.on_network).sum()} more than {config.SNAP_MAX_M} m from a road")
print(facilities.kind.value_counts())

## 4. District boundaries and buildings

District boundaries come from Nominatim (OpenStreetMap's place search). Buildings come from Google Open Buildings v3. The file that covers Assam is large (about 3.5 GB), so we stream it and keep only buildings inside the study box with confidence of at least 0.70. **This cell takes 15 to 30 minutes.**

In [ ]:
def get_boundary(place, tries=4):
    wait = 10
    for attempt in range(1, tries + 1):
        try:
            return ox.geocode_to_gdf(place)
        except Exception as e:
            if attempt == tries:
                raise
            print(f"  lookup failed ({str(e)[:60]}), retrying in {wait}s"); time.sleep(wait); wait *= 2

districts = pd.concat([get_boundary(info["place"]).assign(district=name)[["district", "geometry"]]
                       for name, info in config.DISTRICTS.items()])
districts = gpd.GeoDataFrame(districts, crs="EPSG:4326").to_crs(CRS).reset_index(drop=True)
print(districts.assign(area_km2=districts.area / 1e6).drop(columns="geometry").round(0))

In [ ]:
OB_FILES = ["373", "375"]   # Open Buildings v3 files (S2 cells) that cover the study box
OB_URL = "https://storage.googleapis.com/open-buildings-data/v3/polygons_s2_level_4_gzip/{}_buildings.csv.gz"
BLD_CACHE = f"{RAW}/open_buildings_study_box.parquet"


def load_buildings():
    if os.path.exists(BLD_CACHE):
        return pd.read_parquet(BLD_CACHE)
    parts = []
    for tok in OB_FILES:
        t0, n = time.time(), 0
        for chunk in pd.read_csv(OB_URL.format(tok), usecols=["latitude", "longitude", "area_in_meters", "confidence"],
                                 chunksize=2_000_000, compression="gzip"):
            n += len(chunk)
            keep = chunk[(chunk.longitude >= L) & (chunk.longitude <= R) & (chunk.latitude >= B) &
                         (chunk.latitude <= T) & (chunk.confidence >= config.BUILDING_MIN_CONFIDENCE)]
            parts.append(keep)
        print(f"  file {tok}: read {n:,} buildings in {(time.time() - t0) / 60:.1f} min")
    b = pd.concat(parts, ignore_index=True)
    b.to_parquet(BLD_CACHE)
    return b

buildings = load_buildings()
print(f"{len(buildings):,} buildings in the study box (confidence >= {config.BUILDING_MIN_CONFIDENCE})")

## 5. People per building

WorldPop gives people per 100 m square. Each square's people are shared equally among the buildings inside it. A square with people but no detected building loses its people, so we report how many were lost.

In [ ]:
import rasterio
from rasterio.windows import from_bounds


def allocate_population(b, worldpop_path):
    with rasterio.open(worldpop_path) as src:
        win = from_bounds(L, B, R, T, src.transform).round_offsets().round_lengths()
        pop = src.read(1, window=win).astype(np.float64)
        tr = src.window_transform(win)
        nod = src.nodata
    pop[(pop < 0) | (pop == nod) | ~np.isfinite(pop)] = 0
    col, row = ~tr * (b.longitude.values, b.latitude.values)
    row = np.clip(row.astype(int), 0, pop.shape[0] - 1)
    col = np.clip(col.astype(int), 0, pop.shape[1] - 1)
    cell = row * pop.shape[1] + col
    uniq, inv, counts = np.unique(cell, return_inverse=True, return_counts=True)
    people = pop.ravel()[uniq] / counts
    b = b.copy()
    b["people"] = people[inv]
    captured = pop.ravel()[uniq].sum()
    return b, captured, pop.sum()

buildings, captured, total = allocate_population(buildings, WORLDPOP)
print(f"WorldPop people in the study box: {total:,.0f}; placed on buildings: {captured:,.0f} "
      f"({captured / max(total, 1):.1%})")

## 6. Group buildings into villages and connect them to the roads

Only buildings inside the three districts are grouped. Buildings within 100 m of each other form one settlement (DBSCAN). The settlement point is the average position of its buildings. It is joined to the nearest road junction; if that is more than 500 m away, the settlement is marked "no road access".

In [ ]:
def make_villages(b, districts, G):
    pts = gpd.GeoDataFrame(b, geometry=gpd.points_from_xy(b.longitude, b.latitude), crs="EPSG:4326").to_crs(CRS)
    pts = gpd.sjoin(pts, districts, predicate="within", how="inner").drop(columns="index_right")
    out = []
    for name, grp in pts.groupby("district"):
        xy = np.c_[grp.geometry.x, grp.geometry.y]
        lab = DBSCAN(eps=config.SETTLEMENT_CLUSTER_M, min_samples=1, algorithm="ball_tree").fit_predict(xy)
        g = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "people": grp.people.values, "lab": lab})
        v = g.groupby("lab").agg(x=("x", "mean"), y=("y", "mean"), n_buildings=("x", "size"),
                                 people=("people", "sum")).reset_index(drop=True)
        v["district"] = name
        out.append(v)
        print(f"  {name}: {len(grp):,} buildings -> {len(v):,} settlements")
    v = pd.concat(out, ignore_index=True)
    v = gpd.GeoDataFrame(v, geometry=gpd.points_from_xy(v.x, v.y), crs=CRS).drop(columns=["x", "y"])
    node, dist = ox.distance.nearest_nodes(G, v.geometry.x.values, v.geometry.y.values, return_dist=True)
    v["node"], v["snap_m"] = node, np.round(dist, 1)
    v["no_road_access"] = v["snap_m"] > config.SNAP_MAX_M
    v["people"] = v["people"].round(0)
    v["village_id"] = np.arange(len(v))
    return v

t0 = time.time()
villages = make_villages(buildings, districts, G)
print(f"{len(villages):,} settlements, {villages.no_road_access.sum():,} with no road within "
      f"{config.SNAP_MAX_M} m ({time.time() - t0:.0f} s)")

## 7. Dry run: route every village to its nearest facility with **no flooding**

This gives each village's normal route length, which the flood results will be compared against. It also finds problems in the map before any flooding: a village that cannot reach any facility even on a dry day points to a gap in OpenStreetMap, not a flood. Those villages must be reported separately later, never counted as cut off by the flood.

In [ ]:
def dry_run(G, facilities, villages):
    sources = set(facilities.loc[facilities.on_network, "node"])
    dist = nx.multi_source_dijkstra_path_length(G, sources, weight="length")
    v = villages.copy()
    net = v["node"].map(dist)
    v["normal_route_m"] = (net + v["snap_m"]).round(0)
    v.loc[v.no_road_access, "normal_route_m"] = np.nan
    v["unreachable_dry"] = ~v.no_road_access & net.isna()
    return v

villages = dry_run(G, facilities, villages)

fac_d = gpd.sjoin(facilities, districts, predicate="within").groupby("district").size()
rows = []
for name, v in villages.groupby("district"):
    ok = v[~v.no_road_access & ~v.unreachable_dry]
    rows.append({
        "district": name,
        "settlements": len(v),
        "people": int(v.people.sum()),
        "facilities_in_district": int(fac_d.get(name, 0)),
        "no_road_access": int(v.no_road_access.sum()),
        "no_road_access_people": int(v.loc[v.no_road_access, "people"].sum()),
        "unreachable_even_dry": int(v.unreachable_dry.sum()),
        "unreachable_even_dry_people": int(v.loc[v.unreachable_dry, "people"].sum()),
        "median_route_km": round(ok.normal_route_m.median() / 1000, 1),
        "p90_route_km": round(ok.normal_route_m.quantile(0.9) / 1000, 1),
    })
summary = pd.DataFrame(rows)
summary

## 8. Save, and copy to Google Drive

In [ ]:
strips = edges[["edge_id", "u", "v", "key", "cls", "is_bridge", "geometry"]].copy()
strips["geometry"] = strips.geometry.buffer(config.ROAD_BUFFER_M / 2, cap_style="flat")   # 20 m wide in total

keep_e = ["edge_id", "u", "v", "key", "osmid", "cls", "is_bridge", "length", "name", "geometry"]
e_out = edges[[c for c in keep_e if c in edges.columns]].copy()
for c in ("osmid", "name"):
    if c in e_out.columns:
        e_out[c] = e_out[c].astype(str)
n_out = nodes[["osmid", "x", "y", "geometry"]].copy()

e_out.to_file(f"{OUT}/network.gpkg", layer="edges", driver="GPKG")
n_out.to_file(f"{OUT}/network.gpkg", layer="nodes", driver="GPKG")
strips.to_file(f"{OUT}/network.gpkg", layer="strips", driver="GPKG")
facilities.to_file(f"{OUT}/facilities.gpkg", driver="GPKG")
villages.to_file(f"{OUT}/villages.gpkg", driver="GPKG")
districts.to_file(f"{OUT}/districts.gpkg", driver="GPKG")
summary.to_csv(f"{OUT}/summary_network.csv", index=False)
with open(f"{OUT}/run_info.json", "w") as f:
    json.dump({"date": TODAY, "osm_file": os.path.basename(PBF), "worldpop": WORLDPOP_URL,
               "open_buildings": [OB_URL.format(t) for t in OB_FILES],
               "crs": CRS, "excluded_road_types": sorted(config.EXCLUDED_ROAD_TYPES),
               "building_min_confidence": config.BUILDING_MIN_CONFIDENCE,
               "settlement_cluster_m": config.SETTLEMENT_CLUSTER_M, "snap_max_m": config.SNAP_MAX_M,
               "road_strip_width_m": config.ROAD_BUFFER_M,
               "worldpop_share_placed_on_buildings": round(captured / max(total, 1), 3)}, f, indent=1)
for fn in sorted(os.listdir(OUT)):
    print(f"{fn:25s} {os.path.getsize(f'{OUT}/{fn}') / 1e6:7.1f} MB")

In [ ]:
# Copy everything to Google Drive (folder: My Drive/cutoff_network), then share that folder
from google.colab import drive
import shutil
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/cutoff_network"
shutil.copytree(OUT, dest, dirs_exist_ok=True)
print("Copied to", dest)

## 9. Visual check of 10 rural villages (please do this)

The numbers above only tell us what is in OpenStreetMap. This check tells us whether OpenStreetMap is **complete**. It also finishes the rural village check from Week 1 that was never done.

Run the next cell, then change `i` from 0 to 9. For each village, look at the satellite image: are there roads or tracks leading into the village with **no yellow or red line on top**? Write the answer in the list at the bottom.

In [ ]:
import folium
rng = np.random.default_rng(0)
rural = villages[(villages.n_buildings.between(20, 300)) & ~villages.no_road_access]
sample = rural.iloc[rng.choice(len(rural), size=min(10, len(rural)), replace=False)].to_crs("EPSG:4326")
e_wgs = edges[["cls", "geometry"]]


def show_village(i, radius_m=1500):
    v = sample.iloc[i]
    lat, lon = v.geometry.y, v.geometry.x
    c = gpd.GeoSeries([v.geometry], crs="EPSG:4326").to_crs(CRS).iloc[0]
    near = e_wgs[e_wgs.intersects(c.buffer(radius_m))].to_crs("EPSG:4326")
    m = folium.Map(location=[lat, lon], zoom_start=15, tiles=None)
    folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     attr="Esri World Imagery").add_to(m)
    folium.GeoJson(near[near.cls.isin(config.MAJOR_ROAD_TYPES)], style_function=lambda x: {"color": "red", "weight": 3}).add_to(m)
    folium.GeoJson(near[~near.cls.isin(config.MAJOR_ROAD_TYPES)], style_function=lambda x: {"color": "yellow", "weight": 2}).add_to(m)
    folium.Marker([lat, lon], popup=f"village {v.village_id}, {v.district}, {int(v.people)} people").add_to(m)
    print(f"Village {i}: id {v.village_id}, {v.district}, {v.n_buildings} buildings, about {int(v.people)} people, "
          f"normal route {v.normal_route_m / 1000:.1f} km" if pd.notna(v.normal_route_m) else "")
    return m

i = 0
show_village(i)

In [ ]:
# Fill in after looking: True = all visible roads into the village are mapped, False = some are missing
VISUAL_CHECK = {0: None, 1: None, 2: None, 3: None, 4: None, 5: None, 6: None, 7: None, 8: None, 9: None}
done = [x for x in VISUAL_CHECK.values() if x is not None]
print(f"Checked {len(done)} of 10; complete: {sum(done)}")
pd.DataFrame({"village_id": sample.village_id.values[:len(VISUAL_CHECK)],
              "roads_complete": list(VISUAL_CHECK.values())[:len(sample)]}).to_csv(f"{OUT}/visual_check.csv", index=False)

## Send back to Adityapratap

1. The Google Drive folder `cutoff_network` (shared with him).
2. The summary table from section 7 and the visual check result (how many of 10 villages had complete roads).

**Warning signs to report straight away:**
- `unreachable_even_dry` is more than a few percent of settlements: the network has gaps.
- WorldPop share placed on buildings is below about 80%: many people live where no building was detected.
- Fewer than 6 of 10 villages have complete roads: OpenStreetMap is too incomplete for reliable results in that area.

**Do not use "Save a copy in GitHub" from Colab.** Use *File > Save a copy in Drive*.